# 09 - Explainability

## Objective

This notebook generates SHAP-based global and local explanations for the flight-delay prediction problem using human-readable predictors from the training and validation periods.

The surrogate explainability model supports interpretability analysis for the final report and dashboard without replacing the production Spark ML model saved in notebook 08.

#### Load project configuration and explainability datasets

In [0]:
# Load the project configuration and explainability datasets

from __future__ import annotations

import json

from config import project_config as cfg
from pyspark.sql import functions as F
from utils.model_training import load_hist_modeling_table


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the model-training notebook (07) before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found. "
            "Run the model-training notebook (07) before continuing."
        ) from exc


require_file(cfg.MODEL_FEATURE_MANIFEST_PATH)

feature_manifest = json.loads(
    dbutils.fs.head(cfg.MODEL_FEATURE_MANIFEST_PATH, 1000000)
)

# Use the manifest saved by notebook 07 so explainability stays aligned
# with the persisted hist checkpoints (not a stale in-memory config).
TARGET_COLUMN = feature_manifest["target_column"]
CATEGORICAL_COLUMNS = list(feature_manifest["categorical_columns"])
NUMERICAL_COLUMNS = list(feature_manifest["numerical_columns"])
MODEL_INPUT_COLUMNS = list(feature_manifest["model_input_columns"])

for table_name in [
    cfg.MODELING_TRAIN_HIST_TABLE,
    cfg.MODELING_VALIDATION_HIST_TABLE,
]:
    require_table(table_name)

df_train_hist = load_hist_modeling_table(cfg.MODELING_TRAIN_HIST_TABLE)
df_validation_hist = load_hist_modeling_table(
    cfg.MODELING_VALIDATION_HIST_TABLE
)

manifest_missing = sorted(
    set(MODEL_INPUT_COLUMNS) - set(df_train_hist.columns)
)
if manifest_missing:
    raise RuntimeError(
        "Feature manifest columns are missing from hist checkpoints: "
        f"{manifest_missing}. Re-run notebook 07 after syncing "
        "config/project_config.py and utils/model_training.py."
    )

print("Explainability datasets loaded successfully.")
print(f"Model input columns: {len(MODEL_INPUT_COLUMNS)}")


#### Explainable artificial intelligence (SHAP)

#### Purpose

Although the production prediction model uses Spark's `FeatureHasher` for scalable preprocessing, hashed feature vectors do not preserve a direct mapping between vector positions and the original predictor names. Consequently, the hashed feature space is not suitable for producing human-interpretable SHAP explanations.

To provide meaningful global and local explanations, a separate explainability workflow is created using the original human-readable predictors. A surrogate Logistic Regression model is trained on the explainability dataset using the same target variable as the production model.

The surrogate model is used exclusively for SHAP analysis and does not replace the production prediction model.

This approach enables interpretable feature importance rankings, SHAP summary plots, dependence plots, and individual flight explanations while maintaining consistency with the production prediction pipeline.


#### Prepare the Explainability Dataset

The explainability dataset is created from the training and validation periods only using the human-readable predictor variables.

Unlike the production pipeline, no feature hashing is applied because SHAP requires identifiable feature names in order to produce meaningful explanations.


In [0]:
EXPLAINABILITY_COLUMNS = (
    MODEL_INPUT_COLUMNS
    + [TARGET_COLUMN]
)

explainability_df = (
    df_train_hist
    .unionByName(df_validation_hist)
    .select(*EXPLAINABILITY_COLUMNS)
    .dropna()
)

print(
    f"Explainability rows: "
    f"{explainability_df.count():,}"
)

In [0]:
import pandas as pd

SHAP_SAMPLE_SIZE = cfg.SHAP_SAMPLE_SIZE
SHAP_RANDOM_SEED = cfg.RANDOM_SEED

explainability_sample = (
    explainability_df
    .orderBy(F.rand(seed=SHAP_RANDOM_SEED))
    .limit(SHAP_SAMPLE_SIZE)
)

explainability_pd = explainability_sample.toPandas()

print(f"Explainability sample rows: {len(explainability_pd):,}")
print(f"Explainability columns: {len(explainability_pd.columns)}")

display(explainability_sample.limit(5))

#### Train the Explainability Logistic Regression Model

The sampled explainability dataset is divided into predictor and target variables.

Categorical variables are transformed using one-hot encoding, while numerical variables are passed through without alteration. The resulting feature matrix preserves identifiable feature names so that SHAP values can be mapped back to human-readable predictors.

A scikit-learn Logistic Regression model is then trained solely for explainability. This model supports SHAP analysis and does not replace the production Spark ML prediction model.


In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Separate predictors and target.
X_explainability = explainability_pd[
    MODEL_INPUT_COLUMNS
].copy()

y_explainability = explainability_pd[
    TARGET_COLUMN
].astype(int).copy()


# Scale numerical predictors while preserving sparse compatibility.
numerical_transformer = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(with_mean=False),
        ),
    ]
)


# Configure readable preprocessing.
explainability_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            CATEGORICAL_COLUMNS,
        ),
        (
            "numerical",
            numerical_transformer,
            NUMERICAL_COLUMNS,
        ),
    ],
    remainder="drop",
)


# Configure the explainability Logistic Regression model.
explainability_logistic_regression = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="saga",
    max_iter=5000,
    tol=1e-3,
    random_state=42,
    n_jobs=-1,
)


# Build and train the explainability pipeline.
explainability_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            explainability_preprocessor,
        ),
        (
            "classifier",
            explainability_logistic_regression,
        ),
    ]
)

explainability_pipeline.fit(
    X_explainability,
    y_explainability,
)


# Verify optimizer convergence.
classifier = explainability_pipeline.named_steps[
    "classifier"
]

iterations_used = int(classifier.n_iter_[0])

print(
    "Explainability Logistic Regression model "
    "trained successfully."
)
print(f"Training rows: {len(X_explainability):,}")
print(f"Raw predictors: {len(MODEL_INPUT_COLUMNS)}")
print(f"Iterations used: {iterations_used:,}")
print(
    "Converged:",
    iterations_used
    < explainability_logistic_regression.max_iter,
)

#### Generate SHAP Values

SHAP (SHapley Additive exPlanations) is applied to the explainability Logistic Regression model to quantify how each predictor contributes to the predicted probability of flight delay.

Both global and local explanations are generated.

Global explanations summarize feature importance across many flights, while local explanations explain why a particular flight received its predicted delay probability.

These explanations describe the behavior of the predictive model and should not be interpreted as evidence of causal relationships.


In [0]:
# Obtain encoded feature names after preprocessing.

encoded_feature_names = (
    explainability_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

clean_feature_names = [
    feature
    .replace("categorical__", "")
    .replace("numerical__", "")
    .replace("_", " ")
    for feature in encoded_feature_names
]

print(f"Encoded explainability features: {len(encoded_feature_names):,}")

In [0]:
clean_feature_names = [
    feature
    .replace("categorical__", "")
    .replace("numerical__", "")
    .replace("_", " ")
    for feature in encoded_feature_names
]

DISPLAY_FEATURE_NAMES = {
    "DISTANCE": "Flight Distance",
    "CRS ELAPSED TIME": "Scheduled Elapsed Time",
    "DEP HOUR": "Departure Hour",
    "CRS DEP TIME": "Scheduled Departure Time",
    "CRS ARR TIME": "Scheduled Arrival Time",
    "MONTH": "Month",
    "DAY OF WEEK": "Day of Week",
    "IS WEEKEND": "Weekend Flight",
    "ROUTE HIST DELAY RATE": "Historical Route Delay Rate",
    "AIRLINE HIST DELAY RATE": "Historical Airline Delay Rate",
    "ORIGIN HIST DELAY RATE": "Historical Origin Delay Rate",
    "DEST HIST DELAY RATE": "Historical Destination Delay Rate",
}

clean_feature_names = [
    DISPLAY_FEATURE_NAMES.get(feature, feature)
    for feature in clean_feature_names
]

print("Readable feature names prepared successfully.")

#### Compute SHAP Values

The trained explainability Logistic Regression model is interpreted using SHAP.

The generated SHAP values quantify the contribution of each encoded predictor to the predicted probability of flight delay.

These values are subsequently used to produce:

- Global feature importance
- SHAP summary plots
- Dependence plots
- Local explanations for individual flights displayed in the dashboard.


In [0]:
import shap
import pandas as pd

# ---------------------------------------------------------
# Transform the readable features using the preprocessing
# pipeline.
# ---------------------------------------------------------

X_transformed = (
    explainability_pipeline
    .named_steps["preprocessor"]
    .transform(X_explainability)
)

print("Transformed feature matrix:")
print(X_transformed.shape)

#### Initialize the SHAP Explainer

A SHAP `LinearExplainer` is initialized using the trained explainability Logistic Regression model and the transformed explainability feature matrix.

The explainer estimates the contribution of every encoded predictor to the predicted probability of flight delay. For computational efficiency, SHAP automatically samples a subset of background observations while preserving the overall feature distribution.

This initialization prepares the explainability model for both global and local interpretation.


In [0]:
classifier = (
    explainability_pipeline
    .named_steps["classifier"]
)

explainer = shap.LinearExplainer(
    classifier,
    X_transformed,
)

print("SHAP explainer created successfully.")

#### Compute SHAP Values

SHAP values are computed for every observation in the explainability dataset.

Each SHAP value represents the contribution of an individual encoded predictor toward increasing or decreasing the predicted probability of flight delay for a specific observation.

The resulting explanation matrix is subsequently used to generate global feature importance rankings, SHAP summary plots, dependence plots, and local explanations for individual flights displayed in the dashboard.


In [0]:
shap_values = explainer(X_transformed)

print("SHAP values computed successfully.")
print(shap_values.values.shape)

#### Global Feature Importance

Global SHAP analysis identifies the predictors that contribute most strongly to flight delay predictions across the entire explainability dataset.

The importance ranking is calculated using the mean absolute SHAP value of every encoded feature. Higher values indicate that a predictor has greater influence on the model's predictions.


In [0]:
import numpy as np
import pandas as pd

feature_importance = pd.DataFrame(
    {
        "Feature": clean_feature_names,
        "MeanAbsSHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
    }
)

feature_importance = (
    feature_importance
    .sort_values(
        "MeanAbsSHAP",
        ascending=False,
    )
)

display(
    feature_importance.head(20)
)

#### SHAP Summary Plot

The SHAP summary plot visualizes both the magnitude and direction of feature effects across the explainability dataset.

Each point represents one flight. The horizontal position indicates the SHAP value, while the color represents the corresponding feature value.

Positive SHAP values increase the predicted probability of flight delay, whereas negative SHAP values reduce the predicted probability.


In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

shap.summary_plot(
    shap_values,
    X_transformed,
    feature_names=clean_feature_names,
    max_display=20,
    show=False,
)

plt.tight_layout()
plt.show()

#### Direction of Feature Effects

The direction of each feature's influence is evaluated using the average signed SHAP value.

A positive mean SHAP value indicates that the feature tends to increase predicted delay risk across the explainability sample. A negative mean SHAP value indicates that the feature tends to reduce predicted delay risk.

Because categorical predictors are one-hot encoded, the direction applies to the specific category shown in the feature name. These results describe model behavior and should not be interpreted as causal relationships.


In [0]:
direction_of_effects = pd.DataFrame(
    {
        "Feature": clean_feature_names,
        "Mean_SHAP": shap_values.values.mean(axis=0),
        "Mean_Absolute_SHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
    }
)

direction_of_effects["Effect_Direction"] = np.where(
    direction_of_effects["Mean_SHAP"] > 0,
    "Increases predicted delay risk",
    np.where(
        direction_of_effects["Mean_SHAP"] < 0,
        "Decreases predicted delay risk",
        "Neutral average effect",
    ),
)

direction_of_effects = (
    direction_of_effects
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    direction_of_effects.head(20)
)

#### SHAP Dependence Plots

SHAP dependence plots illustrate how changes in individual predictor values influence the model's predicted probability of flight delay.

Unlike the global summary plot, each dependence plot focuses on a single predictor and shows both the magnitude and direction of its SHAP contribution across the explainability dataset.

The following plots are generated for the most influential operational predictors identified during global SHAP analysis:

- Flight Distance
- Historical Route Delay Rate
- Departure Hour

These plots support deeper interpretation of the model's behavior and help identify how important operational factors influence predicted delay risk.


In [0]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse


DEPENDENCE_FEATURES = [
    "Flight Distance",
    "Historical Route Delay Rate",
    "Departure Hour",
]


for feature_name in DEPENDENCE_FEATURES:

    if feature_name not in clean_feature_names:
        print(f"Skipping {feature_name}: feature not found.")
        continue

    feature_index = clean_feature_names.index(feature_name)

    # Extract only the selected feature column.
    feature_column = X_transformed[:, feature_index]

    if sparse.issparse(feature_column):
        feature_values = feature_column.toarray().ravel()
    else:
        feature_values = np.asarray(feature_column).ravel()

    feature_shap_values = shap_values.values[:, feature_index]

    # Remove any non-finite observations before plotting.
    valid_rows = (
        np.isfinite(feature_values)
        & np.isfinite(feature_shap_values)
    )

    plt.figure(figsize=(9, 6))

    plt.scatter(
        feature_values[valid_rows],
        feature_shap_values[valid_rows],
        alpha=0.35,
        s=18,
    )

    plt.axhline(
        y=0,
        linewidth=1,
        linestyle="--",
    )

    plt.title(
        f"SHAP Dependence Plot: {feature_name}"
    )
    plt.xlabel(f"{feature_name} (standardized value)")
    plt.ylabel("SHAP value (impact on model output)")

    plt.tight_layout()
    plt.show()

#### Local SHAP Explanations

Local SHAP explanations describe how individual predictor variables contributed to the predicted delay probability for a single flight.

Unlike the global explanations, which summarize model behavior across many observations, local explanations identify the specific factors that increased or decreased the predicted delay risk for an individual prediction.

These explanations support transparent decision-making by showing why a particular flight received its predicted delay probability.


In [0]:
import numpy as np
import shap


LOCAL_FLIGHT_INDEX = 0

# Obtain the surrogate model's predicted delay probability.
local_predicted_probability = (
    explainability_pipeline
    .predict_proba(
        X_explainability.iloc[
            [LOCAL_FLIGHT_INDEX]
        ]
    )[0, 1]
)

local_predicted_class = int(
    local_predicted_probability >= 0.50
)

# Build a readable SHAP Explanation object.
local_explanation = shap.Explanation(
    values=shap_values.values[
        LOCAL_FLIGHT_INDEX
    ],
    base_values=shap_values.base_values[
        LOCAL_FLIGHT_INDEX
    ],
    data=(
        X_transformed[
            LOCAL_FLIGHT_INDEX
        ]
        .toarray()
        .ravel()
    ),
    feature_names=clean_feature_names,
)

print(f"Selected flight index: {LOCAL_FLIGHT_INDEX}")
print(
    "Predicted delay probability: "
    f"{local_predicted_probability:.2%}"
)
print(
    "Predicted class: "
    f"{'Delayed' if local_predicted_class == 1 else 'On-time'}"
)

shap.plots.waterfall(
    local_explanation,
    max_display=15,
)

#### Local Explanation Interpretation

The waterfall plot explains the surrogate model's prediction for one selected flight.

Red contributions increased the predicted delay risk, while blue contributions reduced it. The strongest risk-increasing factors for this flight included flight distance, the selected destination, and the historical route delay rate. Scheduled elapsed time, season, weekend status, and other operational characteristics reduced the model's predicted risk.

The SHAP values describe how the model formed this prediction. They do not demonstrate that the listed variables directly caused the flight outcome.


For the selected flight, Flight Distance provided the largest positive contribution toward the predicted delay probability. The selected destination (LGA), destination city (New York, NY), and Historical Route Delay Rate also increased the predicted delay risk.

Conversely, Scheduled Elapsed Time, the Winter season, Weekend Flight status, and the Origin State contributed toward reducing the predicted delay probability.

These SHAP values describe how the model reached its prediction for this individual flight. They should not be interpreted as evidence that the listed variables directly caused the flight outcome.


#### Save SHAP artifacts

Global importance, signed feature effects, a compact SHAP sample, and one local explanation are persisted for the dashboard and final report.

In [0]:
import json


(
    spark.createDataFrame(feature_importance)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
)

(
    spark.createDataFrame(direction_of_effects)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(cfg.SHAP_DIRECTION_EFFECTS_TABLE)
)

top_feature_names = (
    feature_importance
    .sort_values("MeanAbsSHAP", ascending=False)
    .head(cfg.SHAP_VALUES_TOP_FEATURES)["Feature"]
    .tolist()
)
top_feature_indexes = [
    clean_feature_names.index(feature_name)
    for feature_name in top_feature_names
]

sample_row_count = min(
    cfg.SHAP_VALUES_SAMPLE_ROWS,
    shap_values.values.shape[0],
)

shap_sample_rows = []
for row_index in range(sample_row_count):
    for feature_index in top_feature_indexes:
        shap_sample_rows.append(
            {
                "row_index": int(row_index),
                "feature": clean_feature_names[feature_index],
                "shap_value": float(
                    shap_values.values[row_index, feature_index]
                ),
            }
        )

shap_values_sample_df = spark.createDataFrame(shap_sample_rows)

(
    shap_values_sample_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(cfg.SHAP_VALUES_SAMPLE_TABLE)
)

local_explanation_payload = {
    "row_index": int(LOCAL_FLIGHT_INDEX),
    "predicted_delay_probability": float(local_predicted_probability),
    "predicted_class": int(local_predicted_class),
    "top_positive_features": (
        direction_of_effects
        .sort_values("Mean_SHAP", ascending=False)
        .head(5)[["Feature", "Mean_SHAP"]]
        .to_dict(orient="records")
    ),
    "top_negative_features": (
        direction_of_effects
        .sort_values("Mean_SHAP", ascending=True)
        .head(5)[["Feature", "Mean_SHAP"]]
        .to_dict(orient="records")
    ),
}

dbutils.fs.put(
    cfg.SHAP_LOCAL_EXPLANATION_PATH,
    json.dumps(local_explanation_payload, indent=4),
    overwrite=True,
)

print("SHAP artifacts saved successfully.")
print(f"Global importance table: {cfg.SHAP_GLOBAL_IMPORTANCE_TABLE}")
print(f"Direction effects table: {cfg.SHAP_DIRECTION_EFFECTS_TABLE}")
print(f"SHAP sample table: {cfg.SHAP_VALUES_SAMPLE_TABLE}")
print(f"Local explanation file: {cfg.SHAP_LOCAL_EXPLANATION_PATH}")
